### Step 1 — Load encoded MNIST samples

Load the 10 MNIST test images previously encoded with the Spiker data loader. Check the DMA word layout and labels before programming the FPGA.

In [1]:
from pathlib import Path
import numpy as np

folder = Path("/home/xilinx/jupyter_notebooks/snn_sdsp")
bit_file = folder / "snn_sdsp.bit"
hwh_file = folder / "snn_sdsp.hwh"
data_file = folder / "mnist_test10_spikes.npz"

assert bit_file.is_file()
assert hwh_file.is_file()
assert bit_file.stem == hwh_file.stem
assert data_file.is_file()

with np.load(data_file, allow_pickle=False) as data:
    words = np.array(data["words"], copy=True)
    spikes = np.array(data["spikes"], copy=True)
    labels = np.array(data["labels"], copy=True)
    num_steps = int(data["num_steps"])
    gain = float(data["gain"])

print("Words shape:", words.shape)
print("Words dtype:", words.dtype)
print("Spikes shape:", spikes.shape)
print("Labels:", labels.tolist())
print("Time steps:", num_steps)
print("Encoding gain:", gain)
print("First image input spikes:", int(spikes[0].sum()))

assert words.shape == (10, 100, 25)
assert words.dtype == np.uint32
assert spikes.shape == (10, 100, 784)
assert labels.shape == (10,)
assert num_steps == 100

print("MNIST data check passed")

Words shape: (10, 100, 25)
Words dtype: uint32
Spikes shape: (10, 100, 784)
Labels: [7, 2, 1, 0, 4, 1, 4, 9, 5, 9]
Time steps: 100
Encoding gain: 0.1
First image input spikes: 740
MNIST data check passed


### Step 2 — Load the overlay for a fresh MNIST run

Program the FPGA again so the weight memory starts from its initial contents. Bind the SNN registers and DMA driver, then confirm that learning and teacher input are disabled before inference.

In [2]:
from pynq import Overlay, MMIO

print("Programming FPGA...", flush=True)
overlay = Overlay(str(bit_file), download=True)

assert overlay.is_loaded()
assert overlay.ip_dict["axi_dma_mm2s"]["phys_addr"] == 0x40400000
assert overlay.ip_dict["snn_sdsp_0"]["phys_addr"] == 0x43C00000

snn = MMIO(0x43C00000, 0x10000)
dma = overlay.axi_dma_mm2s

control = snn.read(0x00)
status = snn.read(0x04)
ca_interval = snn.read(0x10)
dma_status = dma.mmio.read(0x04)

print("Overlay loaded:", overlay.is_loaded())
print(f"CONTROL: 0x{control:08X}")
print(f"SNN status: 0x{status:08X}")
print("CA interval:", ca_interval)
print(f"DMA status: 0x{dma_status:08X}")
print("DMA running:", dma.sendchannel.running)

assert (control & 0x44) == 0
assert status & 0x01
assert ca_interval == 0
assert dma.sendchannel.running
assert (dma_status & 0x70) == 0

print("Fresh MNIST overlay check passed")

Programming FPGA...


Overlay loaded: True
CONTROL: 0x00000000
SNN status: 0x00000001
CA interval: 0
DMA status: 0x00000000
DMA running: True
Fresh MNIST overlay check passed


### Step 3 — Run one MNIST inference frame

Send MNIST sample 3 through the DMA with learning and teacher input disabled. Check that all 100 time steps are accepted, no stream errors occur, and output spike counts can be read.

In [3]:
import time
from pynq import allocate

def run_inference(index):
    assert 0 <= index < len(words)

    tx = allocate(shape=(2500,), dtype=np.uint32)
    tx[:] = words[index].reshape(-1)
    tx.flush()

    snn.write(0x00, 0x02)
    time.sleep(0.01)
    snn.write(0x00, 0x01)

    dma.sendchannel.transfer(tx)

    deadline = time.monotonic() + 10.0
    while not dma.sendchannel.idle:
        dma_status = dma.mmio.read(0x04)
        if dma_status & 0x70:
            raise RuntimeError(f"DMA error: 0x{dma_status:08X}")
        if time.monotonic() > deadline:
            raise TimeoutError("DMA transfer timeout")
        time.sleep(0.001)

    dma.sendchannel.wait()

    deadline = time.monotonic() + 10.0
    while not (snn.read(0x04) & 0x04):
        if time.monotonic() > deadline:
            raise TimeoutError("SNN completion timeout")
        time.sleep(0.001)

    status = snn.read(0x04)
    accepted = snn.read(0x50)
    counts = [snn.read(0x24 + 4 * i) for i in range(10)]
    result_class = snn.read(0x20)
    dma_status = dma.mmio.read(0x04)

    tx.close()

    assert accepted == 100
    assert (status & 0x18) == 0

    return {
        "dma_status": dma_status,
        "status": status,
        "accepted": accepted,
        "spike_counts": counts,
        "result_class": result_class,
    }

sample_index = 3
result = run_inference(sample_index)

print("Sample index:", sample_index)
print("True label:", int(labels[sample_index]))
print("Input spike count:", int(spikes[sample_index].sum()))
print(f"DMA status: 0x{result['dma_status']:08X}")
print(f"SNN status: 0x{result['status']:08X}")
print("Accepted steps:", result["accepted"])
print("Output spike counts:", result["spike_counts"])
print("Result class register:", result["result_class"])

assert any(count > 0 for count in result["spike_counts"])
print("Single-image MNIST inference passed")

Sample index: 3
True label: 0
Input spike count: 1472
DMA status: 0x00001002
SNN status: 0x00000005
Accepted steps: 100
Output spike counts: [6, 5, 6, 6, 5, 6, 6, 6, 5, 6]
Result class register: 0
Single-image MNIST inference passed


### Step 4 — Record the 10-image inference baseline

Run all 10 encoded MNIST samples with learning and teacher input disabled. Record each image's output spike counts and result register before any supervised training.

In [4]:
baseline_counts = np.zeros((len(words), 10), dtype=np.uint32)
baseline_classes = np.zeros(len(words), dtype=np.uint32)
baseline_input_spikes = np.zeros(len(words), dtype=np.uint32)

for index in range(len(words)):
    frame_result = run_inference(index)

    baseline_counts[index] = frame_result["spike_counts"]
    baseline_classes[index] = frame_result["result_class"]
    baseline_input_spikes[index] = int(spikes[index].sum())

    print(
        f"Index={index:2d} "
        f"Label={int(labels[index])} "
        f"InputSpikes={int(baseline_input_spikes[index]):4d} "
        f"Result={int(baseline_classes[index])} "
        f"Counts={baseline_counts[index].tolist()}"
    )

print("Completed images:", len(words))
print("Baseline inference batch passed")

Index= 0 Label=7 InputSpikes= 740 Result=0 Counts=[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Index= 1 Label=2 InputSpikes=1122 Result=2 Counts=[3, 3, 4, 3, 3, 3, 3, 3, 3, 3]
Index= 2 Label=1 InputSpikes= 365 Result=0 Counts=[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Index= 3 Label=0 InputSpikes=1472 Result=0 Counts=[6, 5, 6, 6, 5, 6, 6, 6, 5, 6]
Index= 4 Label=4 InputSpikes= 747 Result=0 Counts=[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Index= 5 Label=1 InputSpikes= 529 Result=0 Counts=[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Index= 6 Label=4 InputSpikes= 839 Result=0 Counts=[1, 0, 0, 0, 0, 0, 1, 1, 0, 0]
Index= 7 Label=9 InputSpikes= 871 Result=0 Counts=[2, 1, 1, 0, 1, 1, 1, 1, 1, 0]
Index= 8 Label=5 InputSpikes=1190 Result=0 Counts=[4, 4, 4, 3, 4, 4, 4, 4, 4, 4]
Index= 9 Label=9 InputSpikes=1186 Result=0 Counts=[4, 3, 3, 3, 4, 4, 4, 4, 4, 4]
Completed images: 10
Baseline inference batch passed


### Step 5 — Save the inference baseline

Save the labels, input spike counts, output spike counts, and result registers for all 10 images. Record the bitstream hash so this baseline can be matched to the exact hardware build later.

In [5]:
import hashlib

output_file = folder / "mnist_test10_inference_baseline.npz"

if output_file.exists():
    raise FileExistsError(f"Output file already exists: {output_file}")

bit_hash = hashlib.sha256(bit_file.read_bytes()).hexdigest()

np.savez_compressed(
    output_file,
    indices=np.arange(len(words), dtype=np.uint32),
    labels=labels,
    input_spike_counts=baseline_input_spikes,
    output_spike_counts=baseline_counts,
    result_classes=baseline_classes,
    num_steps=np.uint32(num_steps),
    gain=np.float32(gain),
    bit_sha256=np.array(bit_hash),
    mode=np.array("inference_learning_off_teacher_off"),
)

with np.load(output_file, allow_pickle=False) as saved:
    assert np.array_equal(saved["output_spike_counts"], baseline_counts)
    assert np.array_equal(saved["labels"], labels)

print("Saved file:", output_file)
print("File size:", output_file.stat().st_size, "bytes")
print("Bitstream SHA256:", bit_hash)
print("Baseline save check passed")

Saved file: /home/xilinx/jupyter_notebooks/snn_sdsp/mnist_test10_inference_baseline.npz
File size: 2031 bytes
Bitstream SHA256: caadde98ac3f55195118740873910b45e9e2951317da9c699ce84a46d172340f
Baseline save check passed


### Step 6 — Configure the teacher registers

Select MNIST sample 0 and set its true label as the teacher target. Configure the teacher current and keep calcium leak disabled for this first controlled test. Read the values back without starting or enabling learning.

In [6]:
target_index = 0
target_label = int(labels[target_index])
teacher_current_q9 = 128
ca_interval_steps = 0

snn.write(0x0C, target_label)
snn.write(0x18, teacher_current_q9)
snn.write(0x10, ca_interval_steps)
snn.write(0x14, 0)

control = snn.read(0x00)
label_readback = snn.read(0x0C)
current_readback = snn.read(0x18)
ca_readback = snn.read(0x10)

print("Sample index:", target_index)
print("Teacher label:", label_readback)
print("Teacher current Q9:", current_readback)
print("CA interval:", ca_readback)
print(f"CONTROL: 0x{control:08X}")

assert label_readback == target_label
assert current_readback == teacher_current_q9
assert ca_readback == ca_interval_steps
assert (control & 0x44) == 0

print("Teacher register configuration passed")

Sample index: 0
Teacher label: 7
Teacher current Q9: 128
CA interval: 0
CONTROL: 0x00000000
Teacher register configuration passed


### Step 7 — Run the teacher-off learning control

Run MNIST sample 0 with SDSP learning enabled but teacher input disabled. Record the output as a control condition. This run may change weights; the overlay will be reloaded before the teacher-on comparison.

In [7]:
def run_training_frame(index, teacher_enabled):
    tx = allocate(shape=(2500,), dtype=np.uint32)
    tx[:] = words[index].reshape(-1)
    tx.flush()

    snn.write(0x00, 0x02)
    time.sleep(0.01)

    snn.write(0x0C, int(labels[index]))
    snn.write(0x18, teacher_current_q9)
    snn.write(0x10, ca_interval_steps)
    snn.write(0x14, 0)

    control_value = 0x45 if teacher_enabled else 0x05
    snn.write(0x00, control_value)

    dma.sendchannel.transfer(tx)

    deadline = time.monotonic() + 10.0
    while not dma.sendchannel.idle:
        dma_status = dma.mmio.read(0x04)
        if dma_status & 0x70:
            raise RuntimeError(f"DMA error: 0x{dma_status:08X}")
        if time.monotonic() > deadline:
            raise TimeoutError("DMA transfer timeout")
        time.sleep(0.001)

    dma.sendchannel.wait()

    deadline = time.monotonic() + 10.0
    while not (snn.read(0x04) & 0x04):
        if time.monotonic() > deadline:
            raise TimeoutError("SNN completion timeout")
        time.sleep(0.001)

    status = snn.read(0x04)
    accepted = snn.read(0x50)
    counts = [snn.read(0x24 + 4 * i) for i in range(10)]
    result_class = snn.read(0x20)
    control_readback = snn.read(0x00)

    tx.close()

    assert accepted == 100
    assert (status & 0x18) == 0
    assert (control_readback & 0x44) == (control_value & 0x44)

    return {
        "status": status,
        "accepted": accepted,
        "spike_counts": counts,
        "result_class": result_class,
        "control": control_readback,
    }

teacher_off_result = run_training_frame(target_index, teacher_enabled=False)

print("Sample index:", target_index)
print("True label:", target_label)
print(f"CONTROL: 0x{teacher_off_result['control']:08X}")
print(f"SNN status: 0x{teacher_off_result['status']:08X}")
print("Accepted steps:", teacher_off_result["accepted"])
print("Output spike counts:", teacher_off_result["spike_counts"])
print("Result class register:", teacher_off_result["result_class"])
print("Teacher-off control run passed")

Sample index: 0
True label: 7
CONTROL: 0x00000004
SNN status: 0x00000005
Accepted steps: 100
Output spike counts: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Result class register: 0
Teacher-off control run passed


### Step 8 — Restore initial weights

Reload the bitstream to discard any weight changes from the teacher-off control run. Rebind the DMA and SNN drivers, then verify that sample 0 reproduces its saved inference baseline.

In [8]:
print("Reloading bitstream...", flush=True)

overlay = Overlay(str(bit_file), download=True)
snn = MMIO(0x43C00000, 0x10000)
dma = overlay.axi_dma_mm2s

assert overlay.is_loaded()
assert dma.sendchannel.running
assert (snn.read(0x00) & 0x44) == 0

restored_result = run_inference(target_index)
expected_counts = baseline_counts[target_index].tolist()

print("Expected baseline counts:", expected_counts)
print("Restored counts:", restored_result["spike_counts"])
print(f"SNN status: 0x{restored_result['status']:08X}")

assert restored_result["spike_counts"] == expected_counts

print("Initial-weight baseline restored")

Reloading bitstream...
Expected baseline counts: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Restored counts: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
SNN status: 0x00000005
Initial-weight baseline restored


### Step 9 — Run one teacher-enabled training frame

Train on sample 0 with learning and the teacher signal enabled. Record the output spike counts and status. This checks the teacher-enabled path, but does not by itself prove that updated weights were written back.

In [9]:
teacher_on_result = run_training_frame(target_index, teacher_enabled=True)

print("Sample index:", target_index)
print("Teacher label:", target_label)
print("Teacher-off counts:", teacher_off_result["spike_counts"])
print("Teacher-on counts:", teacher_on_result["spike_counts"])
print("Teacher-off target count:", teacher_off_result["spike_counts"][target_label])
print("Teacher-on target count:", teacher_on_result["spike_counts"][target_label])
print(f"CONTROL: 0x{teacher_on_result['control']:08X}")
print(f"SNN status: 0x{teacher_on_result['status']:08X}")
print("Accepted steps:", teacher_on_result["accepted"])

Sample index: 0
Teacher label: 7
Teacher-off counts: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Teacher-on counts: [0, 0, 0, 0, 0, 0, 0, 27, 0, 0]
Teacher-off target count: 0
Teacher-on target count: 27
CONTROL: 0x00000044
SNN status: 0x00000005
Accepted steps: 100


### Step 10 — Teacher-off control group

Reload the bitstream to restore the initial weights. Train on sample 0 ten times with learning enabled and the teacher signal disabled. Then disable learning and measure inference on all ten samples. Keep these results for comparison with the teacher-enabled group.

In [10]:
training_repeats = 10

print("Restoring initial weights for the control group...", flush=True)
overlay = Overlay(str(bit_file), download=True)
snn = MMIO(0x43C00000, 0x10000)
dma = overlay.axi_dma_mm2s

assert overlay.is_loaded()
assert dma.sendchannel.running
assert (snn.read(0x00) & 0x44) == 0

control_training_target_counts = []

for repetition in range(training_repeats):
    result = run_training_frame(target_index, teacher_enabled=False)
    control_training_target_counts.append(
        result["spike_counts"][target_label]
    )

control_probe_results = [
    run_inference(index) for index in range(len(labels))
]

control_probe_counts = np.asarray(
    [result["spike_counts"] for result in control_probe_results],
    dtype=np.uint32,
)
control_probe_classes = np.asarray(
    [result["result_class"] for result in control_probe_results],
    dtype=np.uint32,
)

print("Training repetitions:", training_repeats)
print("Teacher-off target counts during training:",
      control_training_target_counts)
print("Initial sample 0 counts:", baseline_counts[target_index].tolist())
print("Control sample 0 counts:",
      control_probe_counts[target_index].tolist())
print("Samples differing from the initial baseline:",
      int(np.count_nonzero(
          np.any(control_probe_counts != baseline_counts, axis=1)
      )))
print("Teacher-off control group completed")

Restoring initial weights for the control group...
Training repetitions: 10
Teacher-off target counts during training: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Initial sample 0 counts: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Control sample 0 counts: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Samples differing from the initial baseline: 0
Teacher-off control group completed


### Step 11 — Teacher-enabled group

Restore the initial weights. Train on the same sample for the same number of repetitions, this time with the teacher signal enabled. Then disable learning and measure inference on the same ten samples.

In [11]:
print("Restoring initial weights for the teacher-enabled group...", flush=True)
overlay = Overlay(str(bit_file), download=True)
snn = MMIO(0x43C00000, 0x10000)
dma = overlay.axi_dma_mm2s

assert overlay.is_loaded()
assert dma.sendchannel.running
assert (snn.read(0x00) & 0x44) == 0

teacher_training_target_counts = []

for repetition in range(training_repeats):
    result = run_training_frame(target_index, teacher_enabled=True)
    teacher_training_target_counts.append(
        result["spike_counts"][target_label]
    )

teacher_probe_results = [
    run_inference(index) for index in range(len(labels))
]

teacher_probe_counts = np.asarray(
    [result["spike_counts"] for result in teacher_probe_results],
    dtype=np.uint32,
)
teacher_probe_classes = np.asarray(
    [result["result_class"] for result in teacher_probe_results],
    dtype=np.uint32,
)

changed_samples = np.flatnonzero(
    np.any(teacher_probe_counts != control_probe_counts, axis=1)
)

print("Training repetitions:", training_repeats)
print("Teacher-on target counts during training:",
      teacher_training_target_counts)
print("Control sample 0 counts:",
      control_probe_counts[target_index].tolist())
print("Teacher-trained sample 0 counts:",
      teacher_probe_counts[target_index].tolist())
print("Samples differing from the teacher-off control:",
      changed_samples.tolist())

for index in changed_samples:
    print(
        f"Sample {index}: "
        f"control={control_probe_counts[index].tolist()} "
        f"teacher={teacher_probe_counts[index].tolist()}"
    )

print("Teacher-enabled group completed")

Restoring initial weights for the teacher-enabled group...
Training repetitions: 10
Teacher-on target counts during training: [27, 28, 28, 28, 28, 28, 28, 28, 28, 28]
Control sample 0 counts: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Teacher-trained sample 0 counts: [0, 0, 0, 0, 0, 0, 0, 2, 0, 0]
Samples differing from the teacher-off control: [0, 1, 3, 7, 9]
Sample 0: control=[0, 0, 0, 0, 0, 0, 0, 0, 0, 0] teacher=[0, 0, 0, 0, 0, 0, 0, 2, 0, 0]
Sample 1: control=[3, 3, 4, 3, 3, 3, 3, 3, 3, 3] teacher=[3, 3, 4, 3, 3, 3, 3, 4, 3, 3]
Sample 3: control=[6, 5, 6, 6, 5, 6, 6, 6, 5, 6] teacher=[6, 5, 6, 6, 5, 6, 6, 7, 5, 6]
Sample 7: control=[2, 1, 1, 0, 1, 1, 1, 1, 1, 0] teacher=[2, 1, 1, 0, 1, 1, 1, 2, 1, 0]
Sample 9: control=[4, 3, 3, 3, 4, 4, 4, 4, 4, 4] teacher=[4, 3, 3, 3, 4, 4, 4, 5, 4, 4]
Teacher-enabled group completed


### Step 12 — Verify restoration after bitstream reload

Reload the bitstream to restore the initial weights. Run inference on all ten samples and confirm that their output spike counts match the saved initial-weight baseline.

In [12]:
print("Reloading bitstream to restore initial weights...", flush=True)

overlay = Overlay(str(bit_file), download=True)
snn = MMIO(0x43C00000, 0x10000)
dma = overlay.axi_dma_mm2s

assert overlay.is_loaded()
assert dma.sendchannel.running

restored_probe_results = [
    run_inference(index) for index in range(len(labels))
]

restored_probe_counts = np.asarray(
    [result["spike_counts"] for result in restored_probe_results],
    dtype=np.uint32,
)

remaining_differences = np.flatnonzero(
    np.any(restored_probe_counts != baseline_counts, axis=1)
)

print("Sample 0 initial counts:",
      baseline_counts[target_index].tolist())
print("Sample 0 teacher-trained counts:",
      teacher_probe_counts[target_index].tolist())
print("Sample 0 restored counts:",
      restored_probe_counts[target_index].tolist())
print("Samples differing from the initial baseline after reload:",
      remaining_differences.tolist())

assert np.array_equal(restored_probe_counts, baseline_counts)

print("Initial-weight behavior restored after bitstream reload")

Reloading bitstream to restore initial weights...
Sample 0 initial counts: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Sample 0 teacher-trained counts: [0, 0, 0, 0, 0, 0, 0, 2, 0, 0]
Sample 0 restored counts: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Samples differing from the initial baseline after reload: []
Initial-weight behavior restored after bitstream reload


### Step 13 — Verify the saved baseline file

Load the previously saved inference baseline and confirm that it matches the in-memory baseline and the results restored after bitstream reload.

In [13]:
import hashlib

baseline_file = data_file.parent / "mnist_test10_inference_baseline.npz"

with np.load(baseline_file, allow_pickle=False) as saved:
    archived_baseline_counts = saved["output_spike_counts"].copy()
    archived_baseline_classes = saved["result_classes"].copy()
    archived_bit_sha256 = str(saved["bit_sha256"].item())

current_bit_sha256 = hashlib.sha256(bit_file.read_bytes()).hexdigest()

assert np.array_equal(archived_baseline_counts, baseline_counts)
assert np.array_equal(archived_baseline_classes, baseline_classes)
assert np.array_equal(restored_probe_counts, archived_baseline_counts)
assert archived_bit_sha256 == current_bit_sha256

print("Baseline file:", baseline_file)
print("Saved baseline matches the in-memory baseline: True")
print("Restored results match the saved baseline: True")
print("Bitstream hash matches the saved baseline: True")

Baseline file: /home/xilinx/jupyter_notebooks/snn_sdsp/mnist_test10_inference_baseline.npz
Saved baseline matches the in-memory baseline: True
Restored results match the saved baseline: True
Bitstream hash matches the saved baseline: True
